![HOOPS AI](../images/HOOPS_AI.jpg)

# HOOPS AI: Transfer a Face Selection

Select a connected region on the source part and highlight a candidate region on the target. Source and target regions may contain different numbers of faces.

`RegionTransfer` handles region matching between two loaded parts. Results are approximate geometric candidates, not verified manufacturing correspondences.

In [1]:
import hoops_ai
import os
import sys

license_key = os.environ.get("HOOPS_AI_LICENSE")
if not license_key:
    sys.exit("HOOPS_AI_LICENSE environment variable is required.")

hoops_ai.set_license(license_key, validate=True)

------------------------------------------------------------
HOOPS AI
------------------------------------------------------------
  Platform      : Windows 11
  Architecture  : AMD64
  Python        : 3.12.10
------------------------------------------------------------
  Core          : hoops-ai             1.2.0
  CAD Access    : hoops-exchange       26.8.0.dev1  (build: 87762679 2026-09-09T10:20:34Z)
  Conversion    : hoops-converter      26.1.2.dev10  (build: 71b6907 2026-07-24T12:00:12Z)
  Insights      : hoops-web-viewer     0.1.6
------------------------------------------------------------
[OK] HOOPS AI License: Valid


## 1. Load the Parts and Initialize Region Transfer

Run from the `embeddings_pipeline` directory with the HOOPS AI environment and `HOOPS_AI_LICENSE` set. Run once for this pair of files. `RegionPart.load_from_file` encodes one body of a CAD file; pass `body_index` to pick another body of a multi-body file. `RegionTransfer` prepares and aligns both parts once for all later transfers.

In [2]:
import pathlib

from hoops_ai.dataset.region_transfer import Region, RegionPart, RegionTransfer

cad_folder = (pathlib.Path.cwd().parent / "packages" / "cadfiles" / "workflow_transfer").resolve()
cad_filename = cad_folder / "nist_mtc_crada_plate_rev-A.SLDPRT"
cad_filename_target = cad_folder / "nist_stc_09_asme1_ap242-e3.stp"

#cad_filename = cad_folder / "52890_dc92327c_4.stp"
#cad_filename_target = cad_folder / "52890_dc92327c_5.stp"

source_part = RegionPart.load_from_file(cad_filename, body_index=0)
target_part = RegionPart.load_from_file(cad_filename_target, body_index=0)
region_transfer = RegionTransfer(source_part, target_part)

## 2. Open the Source and Target Views

The viewers show the whole file, so use files with a single body.

In [3]:
from hoops_ai.insights import CompareViews

output_dir = (pathlib.Path.cwd() / "out").resolve()

if globals().get("views") is not None:
    views.terminate()
views = CompareViews.start_viewers(cad_filename, cad_filename_target, static_folder=output_dir)

In [4]:
views.show()

## 3. Select, Transfer, and Highlight

Select one or more face regions in the source viewer, then run the next cell. Connected faces are treated as one region; disconnected faces start separate regions. Each source region and its assigned target match receive the same distinct color.

In [5]:
source_faces = views.left.get_selected_faces()

if not source_faces:
    print("Select one or more source faces, then rerun this cell.")
else:
    # One Region per group of connected faces; disconnected faces become separate regions.
    source_regions: list[Region] = region_transfer.split_selection(source_faces)
    multi_result = region_transfer.map_regions(source_regions)
    # Only matched pairs are colored, so every color appears on both parts.
    views.highlight_regions(multi_result.source_regions, multi_result.target_regions, labels=True)
    for index, region in enumerate(multi_result.regions, start=1):
        print(
            f"Region {index}: source {list(region.source_faces)} -> "
            f"target {list(region.target_faces) if region.matched else 'no match (not colored)'}"
        )

Select one or more source faces, then rerun this cell.


## 4. Propose and Map Source Regions Automatically

Find up to 40 whole mechanical features in the source part, such as holes, pockets, slots, text and the outer border. All features are mapped together so that each source feature gets a different target region. A distinctive feature with one clear match acts as an anchor: its nearest source features go to the nearest look-alike target features, in the same distance order. Repeated features keep their relative layout. A feature without a good counterpart stays unmatched rather than taking a wrong target. Regions are shown one at a time: the source feature first, then its target in the same color.

In [6]:
proposals = region_transfer.propose_regions(source_part, limit=40)
multi_result = region_transfer.map_regions(proposals)
views.highlight_regions(multi_result.source_regions, multi_result.target_regions, labels=False)

for rank, (proposal, region) in enumerate(zip(proposals, multi_result.regions, strict=True), start=1):
    print(
        f"{rank}: source={list(region.source_faces)} -> "
        f"target={list(region.target_faces) if region.matched else 'no match (not colored)'}; "
        f"interest={proposal.interest_score:.3f}"
    )
print(f"Colored {multi_result.matched_count} of {len(proposals)} source features.")

C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\3157001957.py:2: UserWarning: Source region 7: approximate transfer stopped with region_limit; the returned region is the best candidate found, not a proven optimum.
  multi_result = region_transfer.map_regions(proposals)
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\3157001957.py:2: UserWarning: Source region 7: no non-overlapping target assignment was available.
  multi_result = region_transfer.map_regions(proposals)
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\3157001957.py:2: UserWarning: Source region 8: approximate transfer stopped with region_limit; the returned region is the best candidate found, not a proven optimum.
  multi_result = region_transfer.map_regions(proposals)
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\3157001957.py:2: UserWarning: Source region 8: no non-overlapping target assignment was available.
  multi_result = region_tr

1: source=[6, 7] -> target=[29, 31]; interest=0.774
2: source=[12, 13] -> target=[28, 30]; interest=0.774
3: source=[8, 9] -> target=[27, 32]; interest=0.774
4: source=[10, 11] -> target=[26, 33]; interest=0.774
5: source=[35, 36, 37, 38] -> target=[59, 60, 61, 62]; interest=0.746
6: source=[18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33] -> target=[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 87, 88, 89, 90]; interest=0.730
7: source=[153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166] -> target=no match (not colored); interest=0.692
8: source=[62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75] -> target=no match (not colored); interest=0.692
9: source=[42, 43, 44, 45] -> target=[70, 71, 72, 73]; interest=0.690
10: source=[46, 47, 48, 49] -> target=[66, 67, 68, 69]; interest=0.690
11: source=[60] -> target=[54]; interest=0.689
12: source=[61] -> target=[53]; interest=0.689
13: source=[56] -> target=[80]; interest=0.689
14: source=[58] -> targe

## 5. Second Case: A New Pair of Parts

Repeat the automatic mapping on a new pair: `nist_ftc_10_asme1_ct5315_rb.CATPart` as source and `nist_ftc_10_asme1_rb_ct5290_ra.CATPart` as target. Review the colored pairs rather than expecting every feature to match. The next cell opens a new pair of viewers, so the viewers above keep their colors. Run sections 1 and 2 first.

In [7]:
cad_filename_second_source = cad_folder / "nist_ftc_10_asme1_ct5315_rb.CATPart"
cad_filename_second_target = cad_folder / "nist_ftc_10_asme1_rb_ct5290_ra.CATPart"


second_source_part = RegionPart.load_from_file(cad_filename_second_source)
second_target_part = RegionPart.load_from_file(cad_filename_second_target)


second_transfer = RegionTransfer(second_source_part, second_target_part)
second_proposals = second_transfer.propose_regions(second_source_part, limit=150)

In [8]:
if globals().get("second_views") is not None:
    second_views.terminate()
second_views = CompareViews.start_viewers(
    cad_filename_second_source, cad_filename_second_target, static_folder=output_dir,
)
second_views.show()

In [9]:
second_result = second_transfer.map_regions(second_proposals)
second_views.highlight_regions(second_result.source_regions, second_result.target_regions, labels=False)

for rank, region in enumerate(second_result.regions, start=1):
    print(
        f"{rank}: source={list(region.source_faces)} -> "
        f"target={list(region.target_faces) if region.matched else 'no match (not colored)'}"
    )
print(f"Colored {second_result.matched_count} of {len(second_proposals)} source features.")

C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\234721429.py:1: UserWarning: Source region 35: approximate transfer stopped with topology_limit; the returned region is the best candidate found, not a proven optimum.
  second_result = second_transfer.map_regions(second_proposals)
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\234721429.py:1: UserWarning: Source region 39: approximate transfer stopped with region_limit; the returned region is the best candidate found, not a proven optimum.
  second_result = second_transfer.map_regions(second_proposals)
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\234721429.py:1: UserWarning: Source region 39: no usable target region was found under the current search settings.
  second_result = second_transfer.map_regions(second_proposals)
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\234721429.py:1: UserWarning: Source region 40: approximate transfer stopped with t

1: source=[282, 283, 298, 299, 300, 301] -> target=[104, 105, 140, 141, 142, 143]
2: source=[284, 285, 302, 303, 304, 305] -> target=[107, 108, 135, 136, 137, 138]
3: source=[286, 287, 306, 307, 308, 309] -> target=[110, 111, 130, 131, 132, 133]
4: source=[280, 281, 294, 295, 296, 297] -> target=[101, 102, 145, 146, 147, 148]
5: source=[4, 5, 6, 8] -> target=[29, 30, 252, 253]
6: source=[37, 38, 58, 59] -> target=[34, 223, 246, 247]
7: source=[43, 44, 60, 61] -> target=[35, 36, 240, 241]
8: source=[45, 46, 62, 63] -> target=[37, 38, 238, 239]
9: source=[0, 1, 2, 3] -> target=[31, 32, 250, 251]
10: source=[35, 36, 56, 57] -> target=[33, 224, 248, 249]
11: source=[257, 258, 291] -> target=[120, 121, 134]
12: source=[255, 256, 290] -> target=[122, 123, 139]
13: source=[253, 254, 289] -> target=[124, 125, 144]
14: source=[259, 260, 292] -> target=[118, 119, 149]
15: source=[182, 183, 184, 185, 237] -> target=[113, 174, 175]
16: source=[186, 187, 188, 189, 238] -> target=[114, 172, 173]
17:

## 6. Extra Example: NIST HTC, CATIA to NX

The same NIST HTC test part exported from two CAD systems: CATIA V5 as source and NX as target. Run sections 1 and 2 first.

In [10]:
htc_catia_filename = cad_folder / "NIST-HTC-PMI-v2" / "nist_htc_V56R2023_20240530.CATPart"
htc_nx_filename = cad_folder / "NIST_HTC_NX2027_20240531.prt"

htc_catia_part = RegionPart.load_from_file(htc_catia_filename)
htc_nx_part = RegionPart.load_from_file(htc_nx_filename)
htc_transfer = RegionTransfer(htc_catia_part, htc_nx_part)
htc_result = htc_transfer.map_regions(htc_transfer.propose_regions(htc_catia_part, limit=40))

for region in htc_result.regions:
    print(list(region.source_faces), "->", list(region.target_faces) if region.matched else "no match")

[100, 101, 121, 122] -> [60, 61]
[102, 103, 114, 115] -> [66, 67]
[104, 105, 112, 113] -> [69, 70]
[98, 99, 123, 124] -> [63, 64]
[106, 107, 110, 111] -> [72, 73]
[52, 53, 97, 125] -> no match
[145, 146, 150, 151] -> [81, 82]
[84, 85] -> [50]
[92, 93] -> no match
[88, 89] -> no match
[86, 87] -> [46]
[90, 91] -> [48]
[116, 117, 118, 119] -> [56, 57]
[56, 57] -> [62]
[54, 55] -> [65]
[64, 65] -> [68]
[66, 67] -> [71]
[68, 69] -> [74]
[82, 83] -> [51]
[60, 61, 96] -> [54, 55]
[95, 108, 109] -> [77, 78]
[34, 48, 49, 152] -> [7, 13, 25, 26]
[46, 47, 154, 155] -> [14, 15, 23, 24]
[128, 129] -> [53]
[74, 75, 76, 77, 139, 140, 141, 142, 143] -> [0, 8, 9, 10, 11, 19, 20, 21, 22]
[80, 81] -> [52]
[78, 79] -> no match
[135, 136] -> no match
[58, 59, 120] -> [58, 59]
[44, 45, 50, 51, 94] -> [2, 16, 17, 18, 27]
[130, 131] -> no match
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] -> [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 11

C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\697398449.py:7: UserWarning: Source region 6: no usable target region was found under the current search settings.
  htc_result = htc_transfer.map_regions(htc_transfer.propose_regions(htc_catia_part, limit=40))
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\697398449.py:7: UserWarning: Source region 9: no usable target region was found under the current search settings.
  htc_result = htc_transfer.map_regions(htc_transfer.propose_regions(htc_catia_part, limit=40))
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\697398449.py:7: UserWarning: Source region 10: no usable target region was found under the current search settings.
  htc_result = htc_transfer.map_regions(htc_transfer.propose_regions(htc_catia_part, limit=40))
C:\Users\LuisSalazar.LY-LS-LEGION\AppData\Local\Temp\ipykernel_12916\697398449.py:7: UserWarning: Source region 27: no usable target region was found under the

In [11]:
if globals().get("htc_views") is not None:
    htc_views.terminate()
htc_views = CompareViews.start_viewers(htc_catia_filename, htc_nx_filename, static_folder=output_dir)
htc_views.show()

In [12]:
htc_views.highlight_regions(htc_result.source_regions, htc_result.target_regions, labels=False)

[(242, 12, 12),
 (73, 122, 242),
 (157, 242, 36),
 (191, 10, 169),
 (57, 191, 169),
 None,
 (191, 117, 29),
 (40, 7, 140),
 None,
 None,
 (46, 140, 42),
 (140, 21, 61),
 (12, 156, 242),
 (228, 242, 73),
 (199, 36, 242),
 (10, 191, 100),
 (191, 85, 57),
 (29, 43, 191),
 (57, 140, 7),
 (140, 42, 108),
 (21, 136, 140),
 (242, 184, 12),
 (150, 73, 242),
 (36, 242, 70),
 (191, 10, 33),
 (57, 114, 191),
 None,
 None,
 (144, 191, 29),
 (140, 7, 140),
 None,
 (42, 140, 111),
 None,
 (140, 70, 21),
 (40, 12, 242),
 (102, 242, 73),
 (242, 36, 131),
 (10, 147, 191),
 (191, 185, 57),
 (136, 29, 191)]

## 7. Extra Example: Landing Gear Design vs Manufacturable (improve name) 

Two parts that must have the same maechanical requirements compare the mechnical region mappings under current region tranfer logic.

In [ ]:
bc_part_regions= cad_folder.joinpath("bc_parts_region")


design_part_file = bc_part_regions.joinpath("landing_gear_design","UPPERSWAY LINK.CATPart")
manufacturable_part_file = bc_part_regions.joinpath("landing_gear_manufacture","Connector.CATPart")


design_part = RegionPart.load_from_file(design_part_file) 
manufacturable_part = RegionPart.load_from_file(manufacturable_part_file)

landing_gear_transfer = RegionTransfer(design_part, manufacturable_part)
region_list = landing_gear_transfer.propose_regions(design_part, limit=40)
landing_gear_result = landing_gear_transfer.map_regions(region_list)



for region in landing_gear_result.regions:
    print(list(region.source_faces), "->", list(region.target_faces) if region.matched else "no match")

In [ ]:
if globals().get("landing_gear_views") is not None:
    landing_gear_views.terminate()
landing_gear_views = CompareViews.start_viewers(design_part_file, manufacturable_part_file, static_folder=output_dir)
landing_gear_views.show()

In [ ]:
landing_gear_views.highlight_regions(landing_gear_result.source_regions, landing_gear_result.target_regions, labels=True)

In [ ]:
landing_gear_faces = landing_gear_views.left.get_selected_faces()

if not landing_gear_faces:
    print("Select one or more source faces, then rerun this cell.")
else:
    # One Region per group of connected faces; disconnected faces become separate regions.
    landing_gear_regions: list[Region] = landing_gear_transfer.split_selection(landing_gear_faces)
    landing_gear_selection_result = landing_gear_transfer.map_regions(landing_gear_regions)
    # Only matched pairs are colored, so every color appears on both parts.
    landing_gear_views.highlight_regions(
        landing_gear_selection_result.source_regions, landing_gear_selection_result.target_regions, labels=True,
    )
    for index, region in enumerate(landing_gear_selection_result.regions, start=1):
        print(
            f"Region {index}: source {list(region.source_faces)} -> "
            f"target {list(region.target_faces) if region.matched else 'no match (not colored)'}"
        )

In [ ]:
print(source_faces)
print(region_transfer.split_selection(source_faces))
print(region_transfer.split_selection([36, 37, 38, 39]))
